In [1]:
#Libraries for handling arrays and measuring time
import numpy as np
import time

In [2]:
#Libraries for loading data and preparing it
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

In [3]:
#KNN model and metrics to check accuracy
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score,classification_report

In [ ]:
print("Downloading Fashion MNIST dataset (this may take 30-60 seconds)....")
fashion_mnist = fetch_openml('Fashion-MNIST', version=1, as_frame=False)

In [ ]:
x = fashion_mnist.data #contains the pixel value
y = fashion_mnist.target.astype(int) 

print(f"Total dataset size: {x.shape[0]} images, each with {x.shape[1]} pixels.")

In [ ]:
#Step1: take a balanced subset of 12,000 images
x_subset, _, y_subset, _ = train_test_split(
    x,y,
    train_size = 12000,
    stratify = y,
    random_state = 42
)

In [ ]:
#Step2: split that subset into training (10,000 images) and testing (2,000 images)
x_train, x_test, y_train, y_test = train_test_split(
    x_subset, y_subset,
    test_size = 2000,
    stratify = y_subset,
    random_state = 42
)

print(f"Training Images: {x_train.shape[0]}")
print(f"Testing Images: {x_test.shape[0]}")

In [ ]:
#Check values before scaling
print(f"Before scaling -> Min: {x_train.min()}, Max: {x_train.max()}")

#Rescale to range [0.0, 0.1]
x_train = x_train/255.0
x_test  = x_test/255.0

#Check values after scaling
print(f"After scaling -> Min: {x_train.min()}, Max: {x_train.max()}")

In [ ]:
#List of K values
k_values = [1,3,5,7,9,15]

#Dictionary to store the results
results = {}

#Print header for the results table
print(f"{'K value':<8} | {'Accuracy':<10} | {'Prediction Time(seconds)':<25}")
print("-"*50)
for k in k_values:
    #1. Initialize the model
    knn = KNeighborsClassifier(n_neighbors = k, metric = 'euclidean', n_jobs = -1)
    #2. "Train" the model (KNN just stores the training data)
    knn.fit(x_train,y_train)
    #3. Predict labels fro test images and measure time
    start_time = time.time()
    y_pred = knn.predict(x_test)
    elapsed_time = time.time() - start_time
    #4. Measure accuracy
    acc =  accuracy_score(y_test,y_pred)
    #Store results
    results[k]={
    "accuracy": acc,
    "time": elapsed_time,
    "prediction": y_pred
    }
    print(f"{k:<8} | {acc*100:<9.2f}% | {elapsed_time:<25.2f}")

In [ ]:
#Readable clothing labels mapped to numbers 0-9
class_names=[
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle Boot"
]
#Find the K with highest accuracy
best_k = max(results, key = lambda k:results[k]["accuracy"])
print(f"Best K is: {best_k} with {results[best_k]['accuracy']*100:.2f}% accuracy\n")
#Print classification breakdown
print("Per-Class Classification Report:")
print(classification_report(y_test,results[best_k]["prediction"], target_names = class_names))